## NBA 3.1 Event List Config (single source of truth)

Loads the NBA 3.1 event lists (market baskets) and defines the helpers that filter claims to those events and apply the clubbing.

* **Used by:** `01_Input_data_creation`, which writes pre-clubbed input files, and every 3.1 patient feature notebook, which uses `event_groups()` as the fixed pivot list and `validate_event_input()` as a guard.
* **Event lists:** `kerendia_nba_3_1_volume/feature_selection/Kerendia NBA 3.1 - {Rx,Dx,Px,Lab} Event List.csv`
  * `EVENT_NAME`: the raw value in the source data (Rx product name, Dx/Px description, Lab test name).
  * `EVENT_CODE`: the raw code (Dx ICD-10, Px CPT/HCPCS, Lab LOINC). Rx has no code column.
  * `EVENT_GROUP`: the feature-level (clubbed) name. Unclubbed rows have `EVENT_GROUP == EVENT_NAME`.
* **Join key:** Rx joins on `EVENT_NAME`. Dx, Px and Lab join on `EVENT_CODE`, because descriptions are cut off at about 100 characters and are not unique.
* **Output convention:** after `apply_event_list()`, the column `EVENT_NAME` holds the `EVENT_GROUP` value. The raw columns (`PRODUCT_NAME`, `DIAGNOSIS_CODE`, ...) are kept for traceability.

Load it with `%run` in its own cell. To change products or codes or clubbing, edit the CSVs only, then re-run `01_Input_data_creation`.

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# ---- NBA 3.1 locations (do not point these at kerendia_nba_3_volume: that holds the 3.0 ops inputs)
VOLUME_31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
EVENT_LIST_DIR = f"{VOLUME_31}/feature_selection"
QUERY_OUTPUT_31 = f"{VOLUME_31}/hcp_level_model/ops_pipeline/query_output"
NOTEBOOK_OUTPUT_31 = f"{VOLUME_31}/hcp_level_model/ops_pipeline/notebook_output"
SNAPSHOT_OUTPUT_31 = f"{VOLUME_31}/hcp_level_model/ops_pipeline_snapshots/data_sources"

EVENT_LIST_FILES = {
    "rx":  f"{EVENT_LIST_DIR}/Kerendia NBA 3.1 - Rx Event List.csv",
    "dx":  f"{EVENT_LIST_DIR}/Kerendia NBA 3.1 - Dx Event List.csv",
    "px":  f"{EVENT_LIST_DIR}/Kerendia NBA 3.1 - Px Event List.csv",
    "lab": f"{EVENT_LIST_DIR}/Kerendia NBA 3.1 - Lab Event List.csv",
}

# Column of the event list used to match claims
EVENT_LIST_JOIN_KEY = {"rx": "EVENT_NAME", "dx": "EVENT_CODE", "px": "EVENT_CODE", "lab": "EVENT_CODE"}

# Rx: the 3.1 product names are pt_lvl_nbrx.PRODUCT_NAME (346/346 match), not LAAD DIM_PRD.PRODUCT_GROUP.
# NDC_CD -> PRODUCT_NAME crosswalk (1:1, ~9.6K NDCs for the 3.1 list) is built from this table.
RX_NDC_CROSSWALK_TABLE = "`ph-com-ai-us-prod-catalog-2553213839599380`.pace_ker_in_house_schema.pt_lvl_nbrx"

# Lab: claims without LOINC_CD fall back to the rule-based TEST_TYPE (GFR / UACR) -> EVENT_GROUP
LAB_TEST_TYPE_FALLBACK = {"GFR": "eGFR TEST (ALL LOINC VARIANTS)", "UACR": "UACR TEST (ALL LOINC VARIANTS)"}

# One event per claim, patient, provider, date and clubbed event (stops double counting when one claim has two member codes)
EVENT_DEDUP_KEYS = ["CLAIM_ID", "PATIENT_ID", "PROVIDER_ID", "SVC_DT", "EVENT_NAME"]

# Pre-clubbed input files written by 01_Input_data_creation (read these in every 3.1 feature notebook)
INPUT_PATHS_31 = {
    "rx_12m":           f"{QUERY_OUTPUT_31}/rx_input_claims_12M_features_input",
    "rx_all":           f"{QUERY_OUTPUT_31}/rx_input_claims_total_events_and_first_occ_features_input",
    # DX: key = content. dx_full_history (no start date) -> all_events file; dx_12m (windowed) -> 12M file
    "dx_full_history":  f"{QUERY_OUTPUT_31}/dx_input_claims_all_events_features_input",
    "dx_12m":           f"{QUERY_OUTPUT_31}/dx_input_claims_12M_and_first_occ_features_input",
    "dx_hosp":          f"{QUERY_OUTPUT_31}/dx_hospitalization_claims_all_events_features_input",
    # PX: no windowed variant exists — px_full_history was a duplicate of px_all (identical query,
    # same row count 673M). Removed px_full_history; use px_all for all First Occ / Recency /
    # Frequency features and px_hosp for Hospitalization.
    "px_all":           f"{QUERY_OUTPUT_31}/px_input_claims_all_events_features_input",
    "px_hosp":          f"{QUERY_OUTPUT_31}/px_hospitalization_claims_all_events_features_input",
    "lab":              f"{QUERY_OUTPUT_31}/lab_input_claims_12M_and_first_occ_features_input",
}

In [0]:
def _norm(c):
    return F.upper(F.trim(c))


def load_event_list(claim_type):
    """Read and validate one 3.1 event list. Adds JOIN_KEY (trimmed, upper-cased join column)."""
    t = claim_type.lower()
    df = pd.read_csv(EVENT_LIST_FILES[t], dtype=str, keep_default_na=False)
    key = EVENT_LIST_JOIN_KEY[t]
    for c in ["EVENT_NAME", "EVENT_GROUP", "IS_CLUBBED", key]:
        assert c in df.columns, f"[{t}] event list is missing column {c}"
    df["JOIN_KEY"] = df[key].str.strip().str.upper()
    assert (df["JOIN_KEY"] != "").all(), f"[{t}] empty {key} in event list"
    conflict = df.groupby("JOIN_KEY")["EVENT_GROUP"].nunique()
    assert (conflict == 1).all(), f"[{t}] keys mapped to more than one EVENT_GROUP: {list(conflict[conflict > 1].index)[:5]}"
    return df.drop_duplicates("JOIN_KEY").reset_index(drop=True)


def event_groups(claim_type):
    """Sorted feature-level event names. Use as the explicit pivot value list in feature notebooks."""
    return sorted(load_event_list(claim_type)["EVENT_GROUP"].unique().tolist())


def event_list_sql_in(claim_type, key=None):
    """Quoted, comma-separated values of the join column, for pushing the filter into the Snowflake query."""
    t = claim_type.lower()
    el = load_event_list(t)
    vals = sorted(el[key or EVENT_LIST_JOIN_KEY[t]].str.strip().unique())
    return ", ".join("'" + v.replace("'", "''") + "'" for v in vals)


def rx_ndc_crosswalk():
    """NDC_CD -> PRODUCT_NAME for the products in the 3.1 Rx list (from pt_lvl_nbrx)."""
    names = load_event_list("rx")["EVENT_NAME"].tolist()
    xw = (spark.table(RX_NDC_CROSSWALK_TABLE)
          .select("NDC_CD", "PRODUCT_NAME").distinct()
          .filter(F.col("PRODUCT_NAME").isin(names)))
    multi = xw.groupBy("NDC_CD").agg(F.countDistinct("PRODUCT_NAME").alias("n")).filter("n > 1").count()
    assert multi == 0, f"{multi} NDCs map to more than one PRODUCT_NAME"
    return xw


def apply_event_list(claims, claim_type, source_col, dedup_keys=None, max_unmatched_share=0.10):
    """Keep only claims whose source_col is in the 3.1 list, and set EVENT_NAME = EVENT_GROUP (clubbing).

    Adds EVENT_IS_CLUBBED. Prints list entries with no claims and fails if more than
    max_unmatched_share of the list is unmatched (wrong source column or stale pull).
    """
    t = claim_type.lower()
    el = load_event_list(t)
    m = spark.createDataFrame(el[["JOIN_KEY", "EVENT_GROUP", "IS_CLUBBED"]])
    if "EVENT_NAME" in claims.columns:
        claims = claims.drop("EVENT_NAME")
    out = (claims.withColumn("JOIN_KEY", _norm(F.col(source_col)))
                 .join(F.broadcast(m), "JOIN_KEY", "inner")
                 .withColumnRenamed("EVENT_GROUP", "EVENT_NAME")
                 .withColumnRenamed("IS_CLUBBED", "EVENT_IS_CLUBBED")
                 .drop("JOIN_KEY"))
    found = {r[0] for r in claims.select(_norm(F.col(source_col))).distinct().collect()}
    missing = el[~el["JOIN_KEY"].isin(found)]
    print(f"[{t}] list entries: {len(el)} | with claims: {len(el) - len(missing)} | without claims: {len(missing)}")
    if len(missing):
        print("   no claims for:", missing[EVENT_LIST_JOIN_KEY[t]].head(25).tolist())
    assert len(missing) / len(el) <= max_unmatched_share, (
        f"[{t}] {len(missing)}/{len(el)} list entries have no claims - check the source column or the pull")
    if dedup_keys:
        out = out.dropDuplicates([k for k in dedup_keys if k in out.columns])
    return out


def validate_event_input(df, claim_type, event_col="EVENT_NAME"):
    """Guard for feature notebooks: every event in the input must be a 3.1 EVENT_GROUP."""
    allowed = set(event_groups(claim_type))
    present = {r[0] for r in df.select(event_col).distinct().collect()}
    extra = present - allowed
    assert not extra, f"[{claim_type}] {event_col} values not in the 3.1 list (stale or unclubbed input?): {sorted(extra)[:10]}"
    print(f"[{claim_type}] OK: {len(present)}/{len(allowed)} EVENT_GROUPs present, none outside the list")
    return sorted(allowed - present)

In [0]:
for _t in ["rx", "dx", "px", "lab"]:
    _el = load_event_list(_t)
    print(f"{_t:>3}: {len(_el):>4} raw entries -> {len(event_groups(_t)):>4} EVENT_GROUPs | clubbed rows: {(_el['IS_CLUBBED'] == 'YES').sum()}")
assert set(LAB_TEST_TYPE_FALLBACK.values()) <= set(event_groups("lab")), "LAB_TEST_TYPE_FALLBACK groups missing from the Lab list"